In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re

/tmp/ipykernel_6193/3676925187.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


fetch data

In [2]:
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

In [3]:
loader = PyPDFLoader("data/kejriwal-et-al-2019-knowledge-graphs-construction-management-and-querying.pdf")
pages = loader.load()

In [4]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

split into chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 100) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

vectorize 

In [6]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [7]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

pass data into llm to disover relationships and nodes,  The entity and relationship types would become labels  and relationship types in the graph

In [8]:
from langchain_ollama import ChatOllama
from langchain_neo4j import Neo4jGraph, LLMGraphTransformer


In [12]:
llm = ChatOllama(model="llama3.1", temperature=0)

In [14]:

graph = Neo4jGraph(url="neo4j://localhost:7687", username="neo4j", password="neo4j123")

ValueError: Could not use APOC procedures. Please ensure the APOC plugin is installed in Neo4j and that 'apoc.meta.data()' is allowed in Neo4j configuration 

In [ ]:

transformer = LLMGraphTransformer(llm=llm)

graph_documents = transformer.convert_to_graph_documents([Document(page_content=text)])
graph.add_graph_documents(graph_documents, baseEntityLabel=True, include_source=True)
